# Week 5 — RAG basics

Goal: index 10–20 public model-risk documents in Chroma and write `search(question)` that returns the 5 most similar chunks with their source file names.

Run this notebook with the `.venv` kernel.

## Step 1 — Collect and convert the documents

List the documents (see `README.md` for the starting set), convert each PDF to text with `pypdf`, and record the page count and word count per document.

In [1]:
from pathlib import Path

# New documents for this week live in docs/; three earlier ones stay where they were downloaded
DOCS = Path("docs")
EXISTING = [
    Path("../week04/docs/SR2602a1.pdf"),
    Path("../week04/docs/OCC-model-risk-booklet-2021.pdf"),
    Path("../week02/pnc-2026.pdf"),
]

pdfs = sorted(DOCS.glob("*.pdf")) + EXISTING
print(len(pdfs), "PDF files")
for p in pdfs:
    print(f"{p.stat().st_size / 1024:8.0f} KB  {p}")

12 PDF files
     959 KB  docs/BAC-DFAST-2026.pdf
    1819 KB  docs/Fed-2026-final-scenarios.pdf
     934 KB  docs/Fed-2026-global-market-shock.pdf
    2718 KB  docs/Fed-2026-macro-model-guide.pdf
     439 KB  docs/JPM-DFAST-2026.pdf
     339 KB  docs/MS-DFAST-2026.pdf
     294 KB  docs/SR11-7-guidance.pdf
      65 KB  docs/SR11-7-letter.pdf
     161 KB  docs/WFC-DFAST-2026.pdf
     208 KB  ../week04/docs/SR2602a1.pdf
    2061 KB  ../week04/docs/OCC-model-risk-booklet-2021.pdf
     541 KB  ../week02/pnc-2026.pdf


## Step 2 — Split into chunks

Chunks of roughly 500–800 tokens with some overlap. `all-MiniLM-L6-v2` reads at most 256 tokens, so check what happens to a longer chunk.

In [2]:
import re
from pypdf import PdfReader
from transformers import AutoTokenizer

CHUNK_TOKENS = 500     # target chunk size, the low end of the plan range, so it fits the 512 tokens the embedding model can take
OVERLAP_TOKENS = 80    # tokens shared by neighbouring chunks, so a sentence cut at one edge is whole in the next

# The same tokenizer as the embedding model, so "tokens" means what the model counts
tokenizer = AutoTokenizer.from_pretrained("sentence-transformers/all-MiniLM-L6-v2", model_max_length=10**7)


def pdf_text(path):
    """All the text of a PDF as one string, without table-of-contents dots and with whitespace collapsed."""
    pages = [page.extract_text() or "" for page in PdfReader(path).pages]
    text = re.sub(r"(\. ?){4,}", " ", " ".join(pages))     # dot leaders of tables of contents ("Planning ...... 24")
    return re.sub(r"\s+", " ", text).strip()


def chunk_text(text, size=CHUNK_TOKENS, overlap=OVERLAP_TOKENS):
    """Split text into windows of `size` tokens that start `size - overlap` tokens apart."""
    enc = tokenizer(text, add_special_tokens=False, return_offsets_mapping=True)
    offsets = enc["offset_mapping"]                       # (start, end) character position of each token
    step = size - overlap
    chunks = []
    for start in range(0, len(offsets), step):
        end = min(start + size, len(offsets))
        chunks.append(text[offsets[start][0]:offsets[end - 1][1]])   # slice the original text, not decoded tokens
        if end == len(offsets):
            break
    return chunks


chunks = []                                               # one dict per chunk
for p in pdfs:
    pieces = chunk_text(pdf_text(p))
    for i, c in enumerate(pieces):
        chunks.append({"source": p.name, "chunk": i, "text": c})
    print(f"{p.name:36s} {len(pieces):4d} chunks")

n_tokens = [len(tokenizer(c["text"], add_special_tokens=False)["input_ids"]) for c in chunks]
print(f"\n{len(chunks)} chunks; tokens per chunk min/median/max: {min(n_tokens)}/{sorted(n_tokens)[len(n_tokens)//2]}/{max(n_tokens)}")
print(f"{sum(n > 512 for n in n_tokens)} chunks are longer than the 512 tokens the embedding model can take")

BAC-DFAST-2026.pdf                     16 chunks
Fed-2026-final-scenarios.pdf           68 chunks
Fed-2026-global-market-shock.pdf       84 chunks
Fed-2026-macro-model-guide.pdf         48 chunks
JPM-DFAST-2026.pdf                     12 chunks
MS-DFAST-2026.pdf                      10 chunks
SR11-7-guidance.pdf                    28 chunks
SR11-7-letter.pdf                       6 chunks
WFC-DFAST-2026.pdf                     12 chunks
SR2602a1.pdf                            9 chunks
OCC-model-risk-booklet-2021.pdf       118 chunks
pnc-2026.pdf                           10 chunks

421 chunks; tokens per chunk min/median/max: 163/500/502
0 chunks are longer than the 512 tokens the embedding model can take


## Step 3 — Embed and store in Chroma

Embed every chunk and store it with its source file name and chunk number as metadata. The database lives in `chroma_db/`.

In [3]:
import chromadb
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")
model.max_seq_length = 512        # default is 256; 512 is the most this model can take, and our chunks are 500 tokens

texts = [c["text"] for c in chunks]
embeddings = model.encode(texts, batch_size=16, normalize_embeddings=True, show_progress_bar=True)
print("embeddings:", embeddings.shape)

# A persistent database in chroma_db/, so the index survives restarting the kernel
client = chromadb.PersistentClient(path="chroma_db", settings=Settings(anonymized_telemetry=False))
try:
    client.delete_collection("model_risk")      # start clean, so running the cell twice does not double the data
except Exception:
    pass                                         # nothing to delete the first time
collection = client.create_collection("model_risk", metadata={"hnsw:space": "cosine"})   # cosine distance, as in Week 4

collection.add(
    ids=[f"{c['source']}::{c['chunk']}" for c in chunks],                                  # unique id per chunk
    documents=texts,                                                                       # the text, returned by searches
    embeddings=embeddings.tolist(),
    metadatas=[{"source": c["source"], "chunk": c["chunk"]} for c in chunks],             # which file and which chunk
)
print("chunks stored in Chroma:", collection.count())

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/27 [00:00<?, ?it/s]

embeddings: (421, 384)
chunks stored in Chroma: 421


In [4]:
len(chunks)

421

## Step 4 — `search(question)`

Return the 5 most similar chunks with their source file names, and try it on a few questions you know the answers to.

In [5]:
def search(question, k=5): 
    """The k chunks most similar to the question, each with its source file name."""
    q = model.encode([question], normalize_embeddings=True)          # embed the question like the chunks
    r = collection.query(query_embeddings=q.tolist(), n_results=k)   # nearest chunks by cosine distance
    return [
        {"source": m["source"], "chunk": m["chunk"], "similarity": 1 - d, "text": t}   # cosine similarity = 1 - distance
        for m, d, t in zip(r["metadatas"][0], r["distances"][0], r["documents"][0])
    ]


def show(question, k=5):
    """Print the search results in a readable form."""
    print(f"\nQ: {question}")
    for rank, hit in enumerate(search(question, k), start=1):
        print(f"{rank}. {hit['source']} (chunk {hit['chunk']}, similarity {hit['similarity']:.2f})")
        print(f"   {hit['text'][:200]}...")


for q in [
    "What is effective challenge?",
    "What is the peak unemployment rate in the severely adverse scenario?",
    "What does SR 26-2 replace?",
]:
    show(q)


Q: What is effective challenge?
1. OCC-model-risk-booklet-2021.pdf (chunk 81, similarity 0.28)
   for effective challenge (e.g., regarding model limitations and assumptions) and approval from qualified personnel. 2. Determine whether the algorithms, mathematical formulas, computer code, software, ...
2. SR11-7-guidance.pdf (chunk 10, similarity 0.28)
   models may not be obvious or intuitive. Model aspects that appear conservative in one model may not be truly conservative compared with alternative methods. For example, simply picking an extreme poin...
3. SR11-7-guidance.pdf (chunk 4, similarity 0.27)
   ifications inappropriate. This is even more of a concern if a model is used outside the environment for which it was designed. Banks may do this intentionally as they apply existing models to new prod...
4. SR11-7-guidance.pdf (chunk 8, similarity 0.26)
   and systematic manner, and are well documented. Models typically are embedded in larger information systems that manage the flow 

## Notes